# Deep-dive — one store, every product

Companion to `00-check_simulated_data.ipynb`. Intended for **small worlds**
(≈20 products, **one store**) so we can render a full per-product card grid
and inspect every decision: lifecycle stage + multiplier, freshness
multiplier, season multiplier, promotion amount, and demand for **every**
product on every step — including when the product is inactive.

Sections:

1. Setup + load (scenario config included for multiplier reconstruction)
2. World view — market supply / demand per region with disruption events overlaid
3. Disruption event log
4. Store financial composition + per-step P&L
5. Store decision summary (all signals on one strip, no transparent lines)
5b. Store total inventory vs capacity (sum of all SKUs)
6. Per-product cards — one figure per SKU with every signal
7. Lifecycle stage heatmap
8. Demand-multiplier-component heatmaps (stage / freshness / season / promo boost)
9. Final snapshot

All line widths and alphas are bumped up vs notebook `00` so faint signals
are visible.

Interactive backend — `%matplotlib widget` (drag to pan, scroll to zoom).

In [ ]:
# Interactive matplotlib + autoreload.
%matplotlib widget
%load_ext autoreload
%autoreload 2

import json
import math
from pathlib import Path
import sys; sys.path.append('../')

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib import cm
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
sns.set_theme(style='whitegrid')
# Bigger default fonts + line widths so nothing is hard to see.
plt.rcParams['figure.dpi'] = 110
plt.rcParams['savefig.dpi'] = 130
plt.rcParams['lines.linewidth'] = 1.8
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 9

## 1. Load

Picks the most recently modified `data/<run>/` folder. Asserts single store
and warns if the catalog is large (this notebook renders one card per
product, so >40 products gets unwieldy).

In [ ]:
repo_root = Path.cwd().resolve().parent
data_root = repo_root / 'data'

sim_dirs = [p for p in data_root.iterdir()
            if p.is_dir() and (p / 'data').is_dir()]
if not sim_dirs:
    raise FileNotFoundError('No simulation output folders found under ../data')

# Pin to llm_world_20 if present (this notebook's target), else latest.
preferred = [p for p in sim_dirs if p.name == 'llm_world_20']
latest_sim_dir = preferred[0] if preferred else max(sim_dirs, key=lambda p: p.stat().st_mtime)
data_dir = latest_sim_dir / 'data'
config_dir = latest_sim_dir / 'config'
print(f'Using simulation folder: {latest_sim_dir}')
print('Data files:', sorted(p.name for p in data_dir.glob('*')))

In [ ]:
products_df = pd.read_parquet(data_dir / 'products.parquet')
stores_df = pd.read_parquet(data_dir / 'stores.parquet')
ts = pd.read_parquet(data_dir / 'timeseries.parquet')

with open(data_dir / 'run_log.json') as f:
    run_log = json.load(f)
with open(config_dir / 'scenario.json') as f:
    scenario = json.load(f)

# Join product static fields into the time-series.
ts = ts.merge(
    products_df[['product_id', 'name', 'category', 'base_price', 'unit_cost',
                 'seasonality', 'freshness_alpha', 'freshness_decay']],
    on='product_id', how='left',
)
ts['simulation_date'] = pd.to_datetime(ts['simulation_date'])
ts['discount_pct'] = (1.0 - ts['price'] / ts['base_price']) * 100.0
ts = ts.sort_values(['store_id', 'product_id', 'simulation_step'])

sim_steps = run_log['global']['time']['simulation_step']
sim_dates = pd.to_datetime(run_log['global']['time']['simulation_date'])
store_ids = sorted(ts['store_id'].unique().tolist())
product_ids = sorted(ts['product_id'].unique().tolist())

print(f'Stores : {store_ids}')
print(f'Steps  : {len(sim_steps)}  ({sim_dates.min().date()} -> {sim_dates.max().date()})')
print(f'Products: {len(product_ids)}')

if len(store_ids) != 1:
    print(f'WARNING: notebook is designed for a single store but found {len(store_ids)}.')
if len(product_ids) > 40:
    print(f'WARNING: {len(product_ids)} products. Per-product card grid will be very long.')

store_id = store_ids[0]
store_row = stores_df[stores_df['store_id'] == store_id].iloc[0]
store_region = store_row['region']
store_total_capacity = float(run_log['stores'][str(store_id)]['step0_capacity'])
print(f'Active store: id={store_id}, region={store_region}, policy={store_row["policy_type"]}')
print(f'Store total inventory capacity: {store_total_capacity:.0f} units')

In [ ]:
# Pull scenario-level multiplier params so we can reconstruct demand math.
market_cfg = scenario['market']
stage_multipliers = market_cfg['stage_multipliers']   # dict: stage -> float
season_months_map = market_cfg['season_months']        # dict: label -> [months]
peak_factor = float(market_cfg['peak_factor'])
off_factor = float(market_cfg['off_factor'])
promo_multiplier = float(market_cfg['promo_multiplier'])
price_elasticity = float(market_cfg['price_elasticity'])
demand_divisor = float(market_cfg['demand_divisor'])
demand_factor_min = float(market_cfg['demand_factor_min'])

print('Stage multipliers   :', stage_multipliers)
print('Peak / off factor   :', peak_factor, off_factor)
print('Promo multiplier    :', promo_multiplier)
print('Price elasticity    :', price_elasticity)
print('Season month map    :', {k: v for k, v in season_months_map.items()})

## 2. World view — market supply / demand + disruptions

Supply and demand per region across the whole run. Disruption events from
`run_log.global.events.occurrences` are overlaid as shaded bands for the
regions they hit; type and severity are annotated.

In [ ]:
# Flatten the event log into a DataFrame of (start_step, end_step, type,
# severity, regions). Each fired event lasts ``duration`` ticks and applies
# to its ``regions`` list.
occurrences = run_log['global']['events']['occurrences']
events = []
for step_idx, ev in enumerate(occurrences):
    if ev is None:
        continue
    events.append({
        'start_step': sim_steps[step_idx],
        'end_step': sim_steps[min(step_idx + int(ev['duration']) - 1, len(sim_steps) - 1)],
        'type': ev['type'],
        'severity': float(ev['severity']),
        'regions': list(ev['regions']),
        'duration': int(ev['duration']),
    })
events_df = pd.DataFrame(events)
print(f'Disruption events fired: {len(events_df)}')
events_df.head()

In [ ]:
# Stable colour per event type for the shading + table.
event_types = sorted({e['type'] for e in events})
type_color = dict(zip(event_types, cm.Set2(np.linspace(0, 1, max(1, len(event_types))))))

regions = list(run_log['global']['market_supply'].keys())
fig, axes = plt.subplots(len(regions), 1, figsize=(12, 3.2 * len(regions)),
                          sharex=True, squeeze=False)
for i, region in enumerate(regions):
    ax = axes[i, 0]
    supply = run_log['global']['market_supply'][region]
    demand = run_log['global']['market_demand'][region]
    ax.plot(sim_steps, supply, color='#1f77b4', lw=2.0, label='Supply', alpha=0.95)
    ax.plot(sim_steps, demand, color='#d62728', lw=2.0, label='Demand', alpha=0.95)
    # Shade disruption windows that hit this region.
    if not events_df.empty:
        for _, ev in events_df.iterrows():
            if region not in ev['regions']:
                continue
            ax.axvspan(ev['start_step'], ev['end_step'],
                       color=type_color[ev['type']], alpha=0.22)
            ax.text(ev['start_step'], ax.get_ylim()[1] * 0.96 if ax.get_ylim()[1] else 1,
                    f"{ev['type']}\n×{ev['severity']:.1f}",
                    fontsize=7, va='top', ha='left',
                    bbox=dict(boxstyle='round,pad=0.15', facecolor='white',
                              edgecolor=type_color[ev['type']], alpha=0.8))
    star = ' (store region)' if region == store_region else ''
    ax.set_title(f'Region: {region}{star}')
    ax.set_ylabel('Units')
    ax.grid(True, alpha=0.35)
    handles, labels = ax.get_legend_handles_labels()
    # Add legend entries for each event type that fired in this region.
    for t in event_types:
        if (not events_df.empty) and any(t == ev['type'] and region in ev['regions']
                                          for _, ev in events_df.iterrows()):
            handles.append(Patch(color=type_color[t], alpha=0.4, label=t))
            labels.append(t)
    ax.legend(handles, labels, loc='upper right')
axes[-1, 0].set_xlabel('Simulation step')
fig.suptitle('Market supply / demand per region (disruption windows shaded)')
fig.tight_layout()
plt.show()

## 3. Disruption event log

In [ ]:
if events_df.empty:
    print('No disruption events fired in this run.')
else:
    display = events_df.copy()
    display['regions'] = display['regions'].apply(lambda r: ', '.join(r))
    display.head(50)
events_df

## 4. Store financial composition

```
equity = cash + inventory_at_cost + outstanding_orders_at_cost
```

Solid alpha and thicker lines so faint regions are still legible.

In [ ]:
cash_balance = run_log['stores'][str(store_id)]['balance']
cash_df = pd.DataFrame({'simulation_step': sim_steps, 'cash': cash_balance,
                        'store_id': store_id})

val = ts.assign(
    inventory_value=ts['inventory'] * ts['unit_cost'],
    outstanding_value=ts['outstanding_orders'] * ts['unit_cost'],
)
store_step = (val.groupby(['store_id', 'simulation_step'], as_index=False)
               .agg(inventory_value=('inventory_value', 'sum'),
                    outstanding_value=('outstanding_value', 'sum'),
                    step_profit=('profit', 'sum'),
                    revenue=('revenue', 'sum'),
                    total_cost=('total_cost', 'sum'),
                    holding_cost=('holding_cost', 'sum')))
store_step = store_step.merge(cash_df, on=['store_id', 'simulation_step'])
store_step['equity'] = store_step['cash'] + store_step['inventory_value'] + store_step['outstanding_value']
store_step = store_step.sort_values('simulation_step')
store_step['cumulative_profit'] = store_step['step_profit'].cumsum()
store_step.tail()

In [ ]:
g = store_step
fig, ax = plt.subplots(figsize=(12, 5))
ax.stackplot(
    g['simulation_step'],
    g['cash'], g['inventory_value'], g['outstanding_value'],
    labels=['Cash', 'Inventory value (at cost)', 'Outstanding orders (at cost)'],
    colors=['#1f77b4', '#2ca02c', '#ff7f0e'], alpha=0.85,
)
ax.plot(g['simulation_step'], g['equity'], color='black', lw=2.2, ls='--',
        label='Total equity')
ax.set_xlabel('Simulation step')
ax.set_ylabel('Equity components')
ax.set_title(f'Store {store_id} — equity composition + cumulative P&L')

ax2 = ax.twinx()
ax2.plot(g['simulation_step'], g['cumulative_profit'], color='crimson', lw=2.2,
         label='Cumulative P&L')
ax2.set_ylabel('Cumulative P&L', color='crimson')
ax2.tick_params(axis='y', labelcolor='crimson')
ax2.grid(False)

h1, l1 = ax.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, loc='upper left')
fig.tight_layout()
plt.show()

In [ ]:
g = store_step
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(g['simulation_step'], g['revenue'], color='#2ca02c', alpha=0.9,
       label='Revenue')
order_cost = g['total_cost'] - g['holding_cost']
ax.bar(g['simulation_step'], -order_cost, color='#d62728', alpha=0.9,
       label='Order cost')
ax.bar(g['simulation_step'], -g['holding_cost'], bottom=-order_cost,
       color='#9467bd', alpha=0.9, label='Holding cost')
ax.plot(g['simulation_step'], g['step_profit'], color='black', lw=2.0,
        marker='o', ms=3.2, label='Step P&L')
ax.axhline(0, color='gray', lw=0.7)
ax.set_xlabel('Simulation step')
ax.set_ylabel('Per-step amount')
ax.set_title(f'Store {store_id} — revenue vs costs per step')
ax.legend(loc='upper left')
fig.tight_layout()
plt.show()

## 5. Store decision summary

Six stacked panels sharing the x-axis: active SKU count, activations/
deactivations, on-promo count + promo starts/ends, orders + outstanding,
median realised price / MSRP. Wide enough to see every step.

In [ ]:
ts_s = ts.copy()
ts_s['prev_active'] = ts_s.groupby(['store_id', 'product_id'])['active_status'].shift()
ts_s['prev_promo'] = ts_s.groupby(['store_id', 'product_id'])['promotion_status'].shift()
is_promo = (ts_s['promotion_status'] != 'Regular Price')
was_promo = (ts_s['prev_promo'] != 'Regular Price') & ts_s['prev_promo'].notna()
ts_s['activated'] = (ts_s['active_status'] == True) & (ts_s['prev_active'] == False)
ts_s['deactivated'] = (ts_s['active_status'] == False) & (ts_s['prev_active'] == True)
ts_s['promo_started'] = is_promo & ~was_promo
ts_s['promo_ended'] = ~is_promo & was_promo
ts_s['on_promo'] = is_promo

decisions = (ts_s.groupby('simulation_step', as_index=False)
              .agg(active_count=('active_status', lambda s: int(s.sum())),
                   activated=('activated', 'sum'),
                   deactivated=('deactivated', 'sum'),
                   on_promo=('on_promo', 'sum'),
                   promo_started=('promo_started', 'sum'),
                   promo_ended=('promo_ended', 'sum'),
                   order_qty=('order_quantity', 'sum'),
                   outstanding=('outstanding_orders', 'sum')))

active_only = ts_s[ts_s['active_status'] == True].copy()
active_only['price_ratio'] = active_only['price'] / active_only['base_price']
pr = (active_only.groupby('simulation_step', as_index=False)
                 .agg(median_price_ratio=('price_ratio', 'median'),
                      mean_price_ratio=('price_ratio', 'mean')))
decisions = decisions.merge(pr, on='simulation_step', how='left')

In [ ]:
fig, axes = plt.subplots(6, 1, figsize=(12, 13), sharex=True)
x = decisions['simulation_step']

axes[0].plot(x, decisions['active_count'], color='#1f77b4', lw=2.0)
axes[0].set_ylabel('Active SKUs')
axes[0].grid(True, alpha=0.4)

axes[1].bar(x, decisions['activated'], color='#2ca02c', alpha=0.95, label='Activated')
axes[1].bar(x, -decisions['deactivated'], color='#d62728', alpha=0.95, label='Deactivated')
axes[1].axhline(0, color='gray', lw=0.6)
axes[1].set_ylabel('Activation events')
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.4)

axes[2].plot(x, decisions['on_promo'], color='#9467bd', lw=2.0, label='SKUs on promo')
axes[2].set_ylabel('On promo')
axes[2].legend(loc='upper right')
axes[2].grid(True, alpha=0.4)

axes[3].bar(x, decisions['promo_started'], color='#9467bd', alpha=0.95, label='Promo start')
axes[3].bar(x, -decisions['promo_ended'], color='#8c564b', alpha=0.95, label='Promo end')
axes[3].axhline(0, color='gray', lw=0.6)
axes[3].set_ylabel('Promo events')
axes[3].legend(loc='upper right')
axes[3].grid(True, alpha=0.4)

axes[4].bar(x, decisions['order_qty'], color='#ff7f0e', alpha=0.95, label='Order qty')
ax4t = axes[4].twinx()
ax4t.plot(x, decisions['outstanding'], color='#1f77b4', lw=2.0, label='Outstanding')
ax4t.grid(False)
axes[4].set_ylabel('Order qty')
ax4t.set_ylabel('Outstanding', color='#1f77b4')
ax4t.tick_params(axis='y', labelcolor='#1f77b4')
h1, l1 = axes[4].get_legend_handles_labels()
h2, l2 = ax4t.get_legend_handles_labels()
axes[4].legend(h1 + h2, l1 + l2, loc='upper right')
axes[4].grid(True, alpha=0.4)

axes[5].plot(x, decisions['median_price_ratio'], color='black', lw=2.0, label='Median')
axes[5].plot(x, decisions['mean_price_ratio'], color='gray', lw=1.5, ls='--', label='Mean')
axes[5].axhline(1.0, color='red', lw=1.0, ls=':')
axes[5].set_ylabel('Price / MSRP\n(active SKUs)')
axes[5].legend(loc='lower left')
axes[5].grid(True, alpha=0.4)

axes[-1].set_xlabel('Simulation step')
fig.suptitle(f'Store {store_id} — decision signals', y=0.995)
fig.tight_layout()
plt.show()

## 5b. Store total inventory vs capacity

On-hand **inventory** summed across **all** SKUs for the active store at each
step, compared to the store's fixed total capacity (`step0_capacity` from the
run log). Outstanding orders are not included.

In [ ]:
inv_by_step = (
    ts[ts['store_id'] == store_id]
    .groupby('simulation_step', as_index=False)['inventory']
    .sum()
    .sort_values('simulation_step')
)
fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(
    inv_by_step['simulation_step'], 0, inv_by_step['inventory'],
    alpha=0.22, color='#1f77b4',
)
ax.plot(
    inv_by_step['simulation_step'], inv_by_step['inventory'],
    color='#1f77b4', lw=2.2, label='Total on-hand inventory (all SKUs)',
)
ax.axhline(
    store_total_capacity, color='black', lw=1.8, ls=':', zorder=5,
    label=f'Store capacity ({store_total_capacity:.0f} units)',
)
ax.set_xlabel('Simulation step')
ax.set_ylabel('Units')
ax.set_title('Store-wide inventory vs capacity')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.4)
fig.tight_layout()
plt.show()

## 6. Per-product cards

One card per SKU with every signal stacked on a shared x-axis:

1. Inventory + outstanding orders + order qty bars
2. Demand vs sales (demand is logged on **every** step regardless of active status — so this panel shows latent demand even when the SKU is deactivated)
3. Price vs base price vs unit cost, with discount % on a right axis
4. **Demand multipliers** — lifecycle-stage, freshness, season, promo boost. Product of these (excluding cross + region demand-factor + price elasticity, which require re-simulation) shown in black.
5. Lifecycle stage as a categorical timeline
6. State strip — active (green) / on promo (purple)
7. Cumulative profit

Cards are sorted by total demand descending so the headline SKUs render first.

In [ ]:
# Per-product lifecycle stage timeline from the run log.
lifecycle_by_pid = {
    pid: run_log['global']['products'][pid]['lifecycle_stage']
    for pid in product_ids
}
# Resolved freshness scalars (Distribution-typed Ware overrides already sampled).
freshness_by_pid = {
    pid: (run_log['global']['products'][pid]['freshness_alpha'],
          run_log['global']['products'][pid]['freshness_decay'])
    for pid in product_ids
}

def stage_mult_series(pid):
    return [stage_multipliers[stage] for stage in lifecycle_by_pid[pid]]

def freshness_mult_series(pid, active_status_series):
    """Reconstruct freshness multiplier per step from active-status edges.

    Mirrors ``src.sim.freshness_curve.multiplier``: at activation time tau=0
    and ``m = 1 + alpha``; tau grows by 1 each step the product stays active.
    Returns 1.0 for steps before any activation has occurred.
    """
    alpha, decay = freshness_by_pid[pid]
    if alpha == 0.0:
        return [1.0] * len(active_status_series)
    out = []
    tau = None  # ticks since last activation; None = never activated
    prev_active = None
    for active in active_status_series:
        if active and (prev_active is None or prev_active is False):
            tau = 0  # activation edge resets tau
        elif active and prev_active:
            tau += 1
        # if inactive we hold tau (curve continues from last activation),
        # matching the freshness clock which is only reset on activate.
        out.append(1.0 + alpha * math.exp(-tau / decay) if tau is not None else 1.0)
        prev_active = active
    return out

def season_mult_series(pid):
    season = products_df.loc[products_df['product_id'] == pid, 'seasonality'].iloc[0]
    peak_months = set(season_months_map.get(season, []))
    return [peak_factor if d.month in peak_months else off_factor for d in sim_dates]

def promo_boost_series(pid_ts):
    on_promo = (pid_ts['promotion_status'] != 'Regular Price').values
    discount = (1.0 - pid_ts['price'].values / pid_ts['base_price'].values)
    return np.where(on_promo, 1.0 + discount * promo_multiplier, 1.0)

# Sort SKUs by total realised demand desc — busiest first.
demand_rank = (ts.groupby('product_id')['demand'].sum()
               .sort_values(ascending=False))
ordered_pids = demand_rank.index.tolist()
print('Top 5 by total demand:')
print(demand_rank.head().to_string())

In [ ]:
# Stage colour map for the categorical lifecycle strip.
all_stages = sorted({s for series in lifecycle_by_pid.values() for s in series})
stage_color = dict(zip(all_stages, cm.viridis(np.linspace(0.1, 0.9, max(1, len(all_stages))))))

def render_product_card(pid):
    g = ts[ts['product_id'] == pid].sort_values('simulation_step').reset_index(drop=True)
    name = g['name'].iloc[0]
    cat = g['category'].iloc[0]
    season = g['seasonality'].iloc[0]
    alpha_f, decay_f = freshness_by_pid[pid]

    # Reconstructed multiplier components on the same step axis.
    stage_m = np.array(stage_mult_series(pid))
    freshness_m = np.array(freshness_mult_series(pid, g['active_status'].tolist()))
    season_m = np.array(season_mult_series(pid))
    promo_b = np.array(promo_boost_series(g))
    combined = stage_m * freshness_m * season_m * promo_b

    stage_series = lifecycle_by_pid[pid]

    fig, axes = plt.subplots(
        7, 1, figsize=(12, 15), sharex=True,
        gridspec_kw={'height_ratios': [3, 3, 3, 3, 1.2, 1.0, 2.2]},
    )
    ax_inv, ax_ds, ax_pr, ax_mult, ax_stage, ax_state, ax_pnl = axes

    # 1. Inventory + outstanding + orders.
    ax_inv.plot(g['simulation_step'], g['inventory'], color='#1f77b4', lw=2.0,
                label='Inventory')
    ax_inv.plot(g['simulation_step'], g['outstanding_orders'], color='#ff7f0e',
                lw=2.0, label='Outstanding')
    ax_inv.bar(g['simulation_step'], g['order_quantity'], color='#2ca02c',
               alpha=0.85, label='Order qty')
    ax_inv.axhline(
        store_total_capacity, color='black', lw=1.8, ls=':', zorder=5,
        label='Store capacity (total)',
    )
    ax_inv.set_ylabel('Units')
    ax_inv.grid(True, alpha=0.4)
    ax_inv.legend(loc='upper right')

    # 2. Demand vs sales (demand shown even when inactive).
    ax_ds.plot(g['simulation_step'], g['demand'], color='#9467bd', lw=2.0,
               ls='--', label='Demand (always sampled)')
    ax_ds.plot(g['simulation_step'], g['sales'], color='#2ca02c', lw=2.0,
               label='Sales')
    ax_ds.fill_between(g['simulation_step'], g['sales'], g['demand'],
                       where=(g['demand'] > g['sales']),
                       color='#d62728', alpha=0.32, label='Unmet')
    # Light grey band on inactive steps so latent-demand context is visible.
    inactive = (~g['active_status']).astype(int).values
    if inactive.any():
        ymax = max(1, int(g['demand'].max()))
        ax_ds.fill_between(g['simulation_step'], 0, ymax * 1.05,
                           where=inactive.astype(bool), step='post',
                           color='lightgray', alpha=0.35, label='Inactive window')
    ax_ds.set_ylabel('Units')
    ax_ds.grid(True, alpha=0.4)
    ax_ds.legend(loc='upper right')

    # 3. Price / base / cost + discount %.
    ax_pr.plot(g['simulation_step'], g['price'], color='black', lw=2.0,
               label='Price')
    ax_pr.plot(g['simulation_step'], g['base_price'], color='gray', lw=1.5,
               ls='--', label='Base price')
    ax_pr.plot(g['simulation_step'], g['unit_cost'], color='red', lw=1.5,
               ls=':', label='Unit cost')
    ax_pr.set_ylabel('Price')
    ax_pr.grid(True, alpha=0.4)
    ax_pr2 = ax_pr.twinx()
    ax_pr2.fill_between(g['simulation_step'], 0, g['discount_pct'],
                        where=(g['discount_pct'] > 0.01), step='post',
                        color='#9467bd', alpha=0.45, label='Discount %')
    ax_pr2.set_ylabel('Discount %', color='#9467bd')
    ax_pr2.tick_params(axis='y', labelcolor='#9467bd')
    ax_pr2.grid(False)
    h1, l1 = ax_pr.get_legend_handles_labels()
    h2, l2 = ax_pr2.get_legend_handles_labels()
    ax_pr.legend(h1 + h2, l1 + l2, loc='upper right')

    # 4. Demand multipliers (stage / freshness / season / promo + combined).
    ax_mult.plot(g['simulation_step'], stage_m, color='#1f77b4', lw=2.0,
                 label='Lifecycle stage')
    ax_mult.plot(g['simulation_step'], freshness_m, color='#2ca02c', lw=2.0,
                 label=f'Freshness (α={alpha_f}, β={decay_f})')
    ax_mult.plot(g['simulation_step'], season_m, color='#ff7f0e', lw=2.0,
                 label=f'Season ({season})')
    ax_mult.plot(g['simulation_step'], promo_b, color='#9467bd', lw=2.0,
                 label='Promo boost')
    ax_mult.plot(g['simulation_step'], combined, color='black', lw=2.4,
                 label='Combined (excl. cross + region + price)')
    ax_mult.axhline(1.0, color='gray', lw=0.8, ls=':')
    ax_mult.set_ylabel('Multiplier')
    ax_mult.grid(True, alpha=0.4)
    ax_mult.legend(loc='upper right', ncol=2)

    # 5. Lifecycle stage categorical strip.
    for t_idx, stage in enumerate(stage_series):
        ax_stage.axvspan(g['simulation_step'].iloc[t_idx] - 0.5,
                         g['simulation_step'].iloc[t_idx] + 0.5,
                         color=stage_color[stage], alpha=0.95)
    ax_stage.set_yticks([])
    ax_stage.set_ylabel('Stage', fontsize=9)
    # Legend in this panel — one patch per stage seen.
    seen_stages = sorted(set(stage_series), key=lambda s: all_stages.index(s))
    ax_stage.legend(
        handles=[Patch(color=stage_color[s], label=s) for s in seen_stages],
        loc='upper right', ncol=max(1, len(seen_stages))
    )

    # 6. Active / promo state strip.
    ax_state.fill_between(g['simulation_step'], 0, g['active_status'].astype(int),
                          step='post', color='#2ca02c', alpha=0.6, label='Active')
    ax_state.fill_between(g['simulation_step'], 0,
                          (g['promotion_status'] != 'Regular Price').astype(int),
                          step='post', color='#9467bd', alpha=0.85, label='On promo')
    ax_state.set_ylim(0, 1.1)
    ax_state.set_yticks([0, 1])
    ax_state.set_ylabel('State')
    ax_state.legend(loc='upper right', ncol=2)

    # 7. Cumulative profit.
    cum_profit = g['profit'].cumsum()
    ax_pnl.plot(g['simulation_step'], cum_profit, color='crimson', lw=2.2)
    ax_pnl.fill_between(g['simulation_step'], 0, cum_profit,
                        where=(cum_profit >= 0), color='#2ca02c', alpha=0.25)
    ax_pnl.fill_between(g['simulation_step'], 0, cum_profit,
                        where=(cum_profit < 0), color='#d62728', alpha=0.25)
    ax_pnl.axhline(0, color='gray', lw=0.7)
    ax_pnl.set_ylabel('Cum. profit')
    ax_pnl.grid(True, alpha=0.4)

    ax_pnl.set_xlabel('Simulation step')
    totals = (f'demand={int(g["demand"].sum())}  sales={int(g["sales"].sum())}  '
              f'profit={g["profit"].sum():.0f}  '
              f'active_steps={int(g["active_status"].sum())}/{len(g)}')
    fig.suptitle(f'{pid}  —  {name}  ({cat})\n{totals}', y=0.995)
    fig.tight_layout()
    plt.show()

In [ ]:
# Render every product, busiest first. Stop earlier by truncating ordered_pids
# (e.g. ordered_pids[:5]) if you only want the headline SKUs.
for pid in ordered_pids:
    render_product_card(pid)

## 7. Lifecycle stage heatmap

Rows = SKUs (sorted by total demand). Columns = simulation steps. Colours
encode lifecycle stage so you can see which products were stuck in decline
vs the ones that rode growth through the run.

In [ ]:
stage_int = {s: i for i, s in enumerate(all_stages)}
grid = np.array([
    [stage_int[s] for s in lifecycle_by_pid[pid]]
    for pid in ordered_pids
])
fig, ax = plt.subplots(figsize=(12, max(3, 0.32 * len(ordered_pids))))
im = ax.imshow(grid, aspect='auto', cmap='viridis',
               interpolation='nearest',
               extent=[sim_steps[0] - 0.5, sim_steps[-1] + 0.5,
                       len(ordered_pids), 0])
ax.set_yticks(np.arange(len(ordered_pids)) + 0.5)
name_map = products_df.set_index('product_id')['name'].to_dict()
ax.set_yticklabels([f'{pid}  {name_map[pid][:32]}' for pid in ordered_pids], fontsize=8)
ax.set_xlabel('Simulation step')
ax.set_title('Lifecycle stage per product over time')
cbar = fig.colorbar(im, ax=ax, ticks=range(len(all_stages)))
cbar.ax.set_yticklabels(all_stages)
fig.tight_layout()
plt.show()

## 8. Demand-multiplier-component heatmaps

Four heatmaps — stage, freshness, season, promo boost. Same row order
(SKU by total demand) so they line up. Diverging colour map around 1.0.

In [ ]:
stage_grid = np.array([stage_mult_series(pid) for pid in ordered_pids])
freshness_grid = np.array([
    freshness_mult_series(pid, ts[ts['product_id'] == pid]
                              .sort_values('simulation_step')['active_status'].tolist())
    for pid in ordered_pids
])
season_grid = np.array([season_mult_series(pid) for pid in ordered_pids])
promo_grid = np.array([
    promo_boost_series(ts[ts['product_id'] == pid].sort_values('simulation_step'))
    for pid in ordered_pids
])

def _plot_grid(grid, title, ax):
    extent = [sim_steps[0] - 0.5, sim_steps[-1] + 0.5, len(ordered_pids), 0]
    span = float(np.abs(grid - 1.0).max() or 0.1)
    im = ax.imshow(grid, aspect='auto', cmap='RdBu_r',
                   vmin=1 - span, vmax=1 + span,
                   interpolation='nearest', extent=extent)
    ax.set_yticks(np.arange(len(ordered_pids)) + 0.5)
    ax.set_yticklabels(ordered_pids, fontsize=7)
    ax.set_title(title)
    return im

fig, axes = plt.subplots(4, 1, figsize=(12, max(10, 0.32 * len(ordered_pids) * 4)),
                          sharex=True)
for ax, grid, title in zip(
    axes,
    [stage_grid, freshness_grid, season_grid, promo_grid],
    ['Stage multiplier', 'Freshness multiplier',
     'Season multiplier', 'Promo boost'],
):
    im = _plot_grid(grid, title, ax)
    fig.colorbar(im, ax=ax)
axes[-1].set_xlabel('Simulation step')
fig.tight_layout()
plt.show()

## 9. Final snapshot

In [ ]:
final_step = ts['simulation_step'].max()
final = ts[ts['simulation_step'] == final_step]
totals = (ts.groupby('product_id', as_index=False)
            .agg(total_demand=('demand', 'sum'),
                 total_sales=('sales', 'sum'),
                 total_revenue=('revenue', 'sum'),
                 total_profit=('profit', 'sum'),
                 active_steps=('active_status', 'sum'),
                 promo_steps=('promotion_status',
                              lambda s: int((s != 'Regular Price').sum()))))
snap = final[['product_id', 'name', 'category', 'inventory',
              'outstanding_orders', 'active_status', 'promotion_status',
              'price']].merge(totals, on='product_id')
snap['fill_rate'] = (snap['total_sales'] / snap['total_demand']
                     .replace(0, np.nan))
snap = snap.sort_values('total_profit', ascending=False).reset_index(drop=True)
snap